# Processamento de Língua Natural - EP

Modelo contextual para classificação de clareza.

A estratégia combina quatro pontos:

- **BERTimbau** como representação contextual em português;
- **classificação ordinal**, aproveitando a ordem natural `c1 < c234 < c5`;
- **características simples de estrutura textual**, calculadas diretamente a partir da resposta;
- **adaptação opcional ao domínio (TAPT)** com os textos do e-SIC sem usar rótulos.

Durante o desenvolvimento, o TAPT pode ficar desligado para reduzir o tempo de execução. O restante do modelo funciona normalmente sem ele.

## Bibliotecas

Dependências principais: Pandas, NumPy, Scikit-learn, PyTorch e Transformers.

Caso o ambiente ainda não tenha as bibliotecas:

```python
%pip install pandas openpyxl numpy scikit-learn torch transformers accelerate tqdm
```

In [ ]:
import gc
import os
import random
import re
import time
from datetime import datetime

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

from transformers import (
    AutoModel,
    AutoModelForMaskedLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    get_linear_schedule_with_warmup,
)

## Configuração do experimento

Este experimento mantém **3 folds e 128 tokens**, mas aumenta o treinamento para **2 épocas**.

A ideia é verificar se o resultado anterior estava sofrendo de underfitting. O TAPT continua desligado para que a comparação com o experimento anterior seja direta.

Os resultados principais também são gravados em `resultados_ep.txt`. O arquivo inclui configuração, métricas por fold, médias, gap de treino-validação, resultado do treino final e distribuição das previsões do teste.

In [ ]:
SEED = 42

MODELO_BASE = "neuralmind/bert-base-portuguese-cased"

# Configuração do experimento atual.
N_SPLITS = 3
MAX_LENGTH = 128
BATCH_SIZE = 8
EPOCHS = 2

LR_BERT = 2e-5
LR_HEAD = 5e-4
WEIGHT_DECAY = 0.01
DROPOUT = 0.20

# Adaptação ao domínio. Continua desligada neste experimento.
USAR_TAPT = False
TAPT_EPOCHS = 1
TAPT_BATCH_SIZE = 8
TAPT_LR = 5e-5
MLM_PROBABILITY = 0.15

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = DEVICE.type == "cuda"

# Arquivo com os resultados principais do experimento.
LOG_PATH = "resultados_ep.txt"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


def log_resultado(mensagem=""):
    mensagem = str(mensagem)
    print(mensagem)

    with open(LOG_PATH, "a", encoding="utf-8") as arquivo:
        arquivo.write(mensagem + "\n")


# Reinicia o arquivo de log a cada execução completa do notebook.
with open(LOG_PATH, "w", encoding="utf-8") as arquivo:
    arquivo.write("EP1 - Classificação de clareza\n")
    arquivo.write("=" * 60 + "\n")
    arquivo.write(
        f"Início: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n"
    )
    arquivo.write(f"Modelo base: {MODELO_BASE}\n")
    arquivo.write(f"Dispositivo: {DEVICE}\n")
    arquivo.write(
        f"Configuração: {N_SPLITS} folds, {EPOCHS} épocas, "
        f"MAX_LENGTH={MAX_LENGTH}, BATCH_SIZE={BATCH_SIZE}\n"
    )
    arquivo.write(f"TAPT: {USAR_TAPT}\n")
    arquivo.write("=" * 60 + "\n\n")

log_resultado(f"Dispositivo: {DEVICE}")
log_resultado(
    f"Configuração: {N_SPLITS} folds, "
    f"{EPOCHS} época(s), máximo de {MAX_LENGTH} tokens"
)
log_resultado(f"Log: {LOG_PATH}")

## Importação dos conjuntos

O conjunto de teste é carregado nesta etapa porque seus textos podem ser usados no TAPT. Os rótulos do teste não são utilizados em nenhum momento.

In [ ]:
dados_treino = pd.read_excel(
    "dados/train.xlsx",
    converters={"resp_text": str, "clarity": str},
)

dados_teste = pd.read_excel(
    "dados/test1.xlsx",
    converters={"resp_text": str, "clarity": str},
)

X_treino = dados_treino["resp_text"].fillna("").astype(str).tolist()
Y_treino = dados_treino["clarity"].fillna("").astype(str).tolist()
X_teste = dados_teste["resp_text"].fillna("").astype(str).tolist()

print("Treino:", len(X_treino))
print("Teste:", len(X_teste))
print()
print(dados_treino["clarity"].value_counts())

## Pré-processamento

Como o BERTimbau foi treinado com texto natural, não removemos stopwords, não aplicamos stemming e não colocamos o texto em minúsculas. A única normalização feita antes da tokenização é de espaços em branco.

As características auxiliares são calculadas sobre o texto original antes dessa normalização.

In [ ]:
def preprocess_text(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


FEATURE_NAMES = [
    "log_caracteres",
    "log_palavras",
    "log_sentencas",
    "tam_medio_palavra",
    "palavras_por_sentenca",
    "diversidade_lexical",
    "proporcao_pontuacao",
    "proporcao_maiusculas",
    "proporcao_digitos",
    "log_quebras_linha",
    "log_urls",
    "log_emails",
    "log_marcadores_lista",
]


def extrair_caracteristicas(text):
    original = str(text)
    normalizado = preprocess_text(original)

    palavras = re.findall(r"\b[\wÀ-ÿ'-]+\b", normalizado, flags=re.UNICODE)
    sentencas = [
        trecho.strip()
        for trecho in re.split(r"[.!?]+", normalizado)
        if trecho.strip()
    ]

    n_chars = max(len(normalizado), 1)
    n_words = max(len(palavras), 1)
    n_sentences = max(len(sentencas), 1)

    letras = [c for c in original if c.isalpha()]
    n_letters = max(len(letras), 1)

    pontuacao = sum(c in ".,;:!?()[]{}-–—/\\\"" for c in original)
    maiusculas = sum(c.isupper() for c in letras)
    digitos = sum(c.isdigit() for c in original)

    urls = len(re.findall(r"https?://\S+|www\.\S+", original, flags=re.IGNORECASE))
    emails = len(
        re.findall(
            r"\b[\w\.-]+@[\w\.-]+\.\w+\b",
            original,
            flags=re.IGNORECASE,
        )
    )

    marcadores_lista = len(
        re.findall(
            r"(?m)^\s*(?:[-*•]|\d+[.)])\s+",
            original,
        )
    )

    diversidade = len({p.lower() for p in palavras}) / n_words
    tam_medio_palavra = sum(len(p) for p in palavras) / n_words
    palavras_por_sentenca = len(palavras) / n_sentences

    return np.array(
        [
            np.log1p(len(normalizado)),
            np.log1p(len(palavras)),
            np.log1p(len(sentencas)),
            tam_medio_palavra,
            palavras_por_sentenca,
            diversidade,
            pontuacao / n_chars,
            maiusculas / n_letters,
            digitos / n_chars,
            np.log1p(original.count("\n")),
            np.log1p(urls),
            np.log1p(emails),
            np.log1p(marcadores_lista),
        ],
        dtype=np.float32,
    )


features_treino = np.vstack(
    [extrair_caracteristicas(texto) for texto in X_treino]
)

features_teste = np.vstack(
    [extrair_caracteristicas(texto) for texto in X_teste]
)

pd.DataFrame(features_treino, columns=FEATURE_NAMES).describe().T

## Codificação dos rótulos

Os três rótulos são convertidos para uma escala ordinal:

- `c1` → 0
- `c234` → 1
- `c5` → 2

O modelo não aprende três classes independentes. Ele aprende dois limiares:

1. a resposta está acima de `c1`?
2. a resposta está acima de `c234`?

Assim, a própria arquitetura preserva a ordem das classes.

In [ ]:
LABEL_TO_INDEX = {
    "c1": 0,
    "c234": 1,
    "c5": 2,
}

INDEX_TO_LABEL = {
    0: "c1",
    1: "c234",
    2: "c5",
}

labels_invalidos = sorted(set(Y_treino) - set(LABEL_TO_INDEX))

if labels_invalidos:
    raise ValueError(f"Rótulos inesperados no treino: {labels_invalidos}")

y_indices = np.array(
    [LABEL_TO_INDEX[label] for label in Y_treino],
    dtype=np.int64,
)


def gerar_alvos_ordinais(labels):
    labels = labels.long()

    return torch.stack(
        [
            labels > 0,
            labels > 1,
        ],
        dim=1,
    ).float()


def calcular_pos_weight(labels):
    targets = np.column_stack(
        [
            labels > 0,
            labels > 1,
        ]
    ).astype(np.float32)

    positivos = targets.sum(axis=0)
    negativos = len(targets) - positivos

    pesos = np.divide(
        negativos,
        positivos,
        out=np.ones_like(negativos, dtype=np.float32),
        where=positivos > 0,
    )

    return torch.tensor(pesos, dtype=torch.float32)

## Dataset do classificador

A tokenização é feita com o tokenizador do próprio BERTimbau. As características estruturais entram como um vetor separado e são concatenadas à representação contextual dentro do modelo.

In [ ]:
class ClarezaDataset(Dataset):
    def __init__(
        self,
        textos,
        features,
        tokenizer,
        labels=None,
        max_length=128,
    ):
        textos = [preprocess_text(texto) for texto in textos]

        # O padding será feito por batch para evitar processar
        # tokens vazios desnecessariamente.
        self.tokens = tokenizer(
            textos,
            padding=False,
            truncation=True,
            max_length=max_length,
            add_special_tokens=True,
        )

        self.features = torch.tensor(
            features,
            dtype=torch.float32,
        )

        self.labels = None

        if labels is not None:
            self.labels = torch.tensor(
                labels,
                dtype=torch.long,
            )

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        item = {
            key: self.tokens[key][idx]
            for key in self.tokens
        }

        item["features"] = self.features[idx]

        if self.labels is not None:
            item["labels"] = self.labels[idx]

        return item


def criar_collate_fn(tokenizer):
    def collate_fn(batch):
        token_batch = []

        for item in batch:
            token_item = {
                key: item[key]
                for key in item
                if key not in {"features", "labels"}
            }
            token_batch.append(token_item)

        padded = tokenizer.pad(
            token_batch,
            padding=True,
            return_tensors="pt",
        )

        padded["features"] = torch.stack(
            [item["features"] for item in batch]
        )

        if "labels" in batch[0]:
            padded["labels"] = torch.stack(
                [item["labels"] for item in batch]
            )

        return padded

    return collate_fn

## Adaptação ao domínio (TAPT)

Esta etapa é opcional.

O BERTimbau já conhece português, mas não foi treinado especificamente com respostas de órgãos públicos. No TAPT, o modelo continua o pré-treinamento por *masked language modeling* usando textos do próprio domínio.

Na validação cruzada, o fold de validação fica de fora dessa etapa. São usados apenas:

- os textos de treino daquele fold;
- `test1.xlsx`, sem qualquer rótulo.

Isso evita usar o texto da validação na adaptação do modelo.

In [ ]:
class MLMDataset(Dataset):
    def __init__(self, textos, tokenizer, max_length):
        textos = [preprocess_text(texto) for texto in textos]

        self.tokens = tokenizer(
            textos,
            truncation=True,
            max_length=max_length,
            add_special_tokens=True,
        )

    def __len__(self):
        return len(self.tokens["input_ids"])

    def __getitem__(self, idx):
        return {
            key: torch.tensor(value[idx], dtype=torch.long)
            for key, value in self.tokens.items()
        }


def executar_tapt(textos, output_dir):
    tokenizer = AutoTokenizer.from_pretrained(MODELO_BASE)

    dataset = MLMDataset(
        textos=textos,
        tokenizer=tokenizer,
        max_length=MAX_LENGTH,
    )

    collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=True,
        mlm_probability=MLM_PROBABILITY,
    )

    model = AutoModelForMaskedLM.from_pretrained(MODELO_BASE)

    args = TrainingArguments(
        output_dir=output_dir,
        overwrite_output_dir=True,
        num_train_epochs=TAPT_EPOCHS,
        per_device_train_batch_size=TAPT_BATCH_SIZE,
        learning_rate=TAPT_LR,
        weight_decay=WEIGHT_DECAY,
        save_strategy="no",
        logging_steps=50,
        report_to="none",
        fp16=torch.cuda.is_available(),
        seed=SEED,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=dataset,
        data_collator=collator,
    )

    trainer.train()

    os.makedirs(output_dir, exist_ok=True)
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)

    del trainer
    del model

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return output_dir

## Classificador ordinal

O BERT gera uma representação contextual para cada resposta. Essa representação é combinada com as características estruturais.

Em vez de três logits independentes, o modelo produz um único escore de clareza e aprende dois pontos de corte ordenados. O segundo ponto de corte é parametrizado de forma a ser sempre maior que o primeiro.

In [ ]:
class BertOrdinalClareza(nn.Module):
    def __init__(
        self,
        model_name,
        n_features,
        dropout=0.20,
    ):
        super().__init__()

        self.bert = AutoModel.from_pretrained(model_name)

        hidden_size = self.bert.config.hidden_size

        self.feature_net = nn.Sequential(
            nn.Linear(n_features, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        self.score = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size + 32, 1),
        )

        self.cutpoint_1 = nn.Parameter(
            torch.tensor(-0.5, dtype=torch.float32)
        )

        self.cutpoint_delta = nn.Parameter(
            torch.tensor(1.0, dtype=torch.float32)
        )

    def forward(
        self,
        input_ids,
        attention_mask,
        features,
        token_type_ids=None,
    ):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )

        cls = outputs.last_hidden_state[:, 0, :]

        feature_repr = self.feature_net(features)

        combined = torch.cat(
            [cls, feature_repr],
            dim=1,
        )

        score = self.score(combined)

        cutpoint_1 = self.cutpoint_1
        cutpoint_2 = (
            self.cutpoint_1
            + F.softplus(self.cutpoint_delta)
        )

        logits = torch.cat(
            [
                score - cutpoint_1,
                score - cutpoint_2,
            ],
            dim=1,
        )

        return logits

## Treinamento e predição

A taxa de aprendizado do BERT é menor que a taxa usada na cabeça ordinal. Isso permite ajustar a representação contextual sem alterar agressivamente os pesos pré-treinados.

A loss usa os dois alvos ordinais. Os pesos positivos são calculados dentro de cada fold para compensar o desbalanceamento.

In [ ]:
def criar_otimizador(model):
    bert_param_ids = {
        id(param)
        for param in model.bert.parameters()
    }

    bert_params = []
    head_params = []

    for param in model.parameters():
        if not param.requires_grad:
            continue

        if id(param) in bert_param_ids:
            bert_params.append(param)
        else:
            head_params.append(param)

    return torch.optim.AdamW(
        [
            {
                "params": bert_params,
                "lr": LR_BERT,
            },
            {
                "params": head_params,
                "lr": LR_HEAD,
            },
        ],
        weight_decay=WEIGHT_DECAY,
    )


def treinar_modelo(
    model,
    dataset,
    tokenizer,
    pos_weight,
    epochs,
):
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=criar_collate_fn(tokenizer),
        pin_memory=PIN_MEMORY,
    )

    optimizer = criar_otimizador(model)

    total_steps = len(loader) * epochs
    warmup_steps = int(total_steps * 0.10)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=pos_weight.to(DEVICE)
    )

    model.to(DEVICE)

    historico_loss = []

    for epoch in range(epochs):
        model.train()
        losses = []

        barra = tqdm(
            loader,
            desc=f"Época {epoch + 1}/{epochs}",
            leave=True,
        )

        for batch in barra:
            optimizer.zero_grad()

            labels = batch["labels"].to(
                DEVICE,
                non_blocking=PIN_MEMORY,
            )
            features = batch["features"].to(
                DEVICE,
                non_blocking=PIN_MEMORY,
            )
            input_ids = batch["input_ids"].to(
                DEVICE,
                non_blocking=PIN_MEMORY,
            )
            attention_mask = batch["attention_mask"].to(
                DEVICE,
                non_blocking=PIN_MEMORY,
            )

            token_type_ids = batch.get("token_type_ids")

            if token_type_ids is not None:
                token_type_ids = token_type_ids.to(
                    DEVICE,
                    non_blocking=PIN_MEMORY,
                )

            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
                features=features,
            )

            targets = gerar_alvos_ordinais(labels).to(DEVICE)

            loss = criterion(logits, targets)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0,
            )

            optimizer.step()
            scheduler.step()

            losses.append(loss.item())
            barra.set_postfix(loss=f"{loss.item():.4f}")

        media_loss = float(np.mean(losses))
        historico_loss.append(media_loss)

        log_resultado(
            f"Época {epoch + 1}/{epochs} "
            f"- loss média: {media_loss:.4f}"
        )

    return historico_loss


@torch.inference_mode()
def prever_indices(
    model,
    dataset,
    tokenizer,
    descricao="Predição",
):
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=criar_collate_fn(tokenizer),
        pin_memory=PIN_MEMORY,
    )

    model.eval()
    model.to(DEVICE)

    predicoes = []

    for batch in tqdm(
        loader,
        desc=descricao,
        leave=False,
    ):
        features = batch["features"].to(
            DEVICE,
            non_blocking=PIN_MEMORY,
        )
        input_ids = batch["input_ids"].to(
            DEVICE,
            non_blocking=PIN_MEMORY,
        )
        attention_mask = batch["attention_mask"].to(
            DEVICE,
            non_blocking=PIN_MEMORY,
        )

        token_type_ids = batch.get("token_type_ids")

        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(
                DEVICE,
                non_blocking=PIN_MEMORY,
            )

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            features=features,
        )

        probabilidades_cumulativas = torch.sigmoid(logits)

        pred = (
            probabilidades_cumulativas > 0.5
        ).sum(dim=1)

        predicoes.extend(
            pred.cpu().numpy().tolist()
        )

    return np.array(
        predicoes,
        dtype=np.int64,
    )

## Validação cruzada

Mantemos a mesma lógica do baseline: validação estratificada e comparação entre acurácia de treino e validação.

O `StandardScaler` é ajustado apenas com as instâncias de treino de cada fold. Isso evita vazamento das características auxiliares.

In [ ]:
cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED,
)

resultados = []
inicio_cv = time.time()

for fold, (train_idx, val_idx) in enumerate(
    cv.split(X_treino, y_indices),
    start=1,
):
    inicio_fold = time.time()

    log_resultado()
    log_resultado("=" * 60)
    log_resultado(f"FOLD {fold}/{N_SPLITS}")
    log_resultado("=" * 60)

    textos_fold_treino = [
        X_treino[i]
        for i in train_idx
    ]

    textos_fold_validacao = [
        X_treino[i]
        for i in val_idx
    ]

    y_fold_treino = y_indices[train_idx]
    y_fold_validacao = y_indices[val_idx]

    scaler = StandardScaler()

    features_fold_treino = scaler.fit_transform(
        features_treino[train_idx]
    )

    features_fold_validacao = scaler.transform(
        features_treino[val_idx]
    )

    model_name = MODELO_BASE

    if USAR_TAPT:
        textos_tapt = (
            textos_fold_treino
            + X_teste
        )

        model_name = executar_tapt(
            textos=textos_tapt,
            output_dir=f"modelos/tapt_fold_{fold}",
        )

    tokenizer = AutoTokenizer.from_pretrained(
        model_name
    )

    dataset_fold_treino = ClarezaDataset(
        textos=textos_fold_treino,
        features=features_fold_treino,
        labels=y_fold_treino,
        tokenizer=tokenizer,
        max_length=MAX_LENGTH,
    )

    dataset_fold_validacao = ClarezaDataset(
        textos=textos_fold_validacao,
        features=features_fold_validacao,
        labels=y_fold_validacao,
        tokenizer=tokenizer,
        max_length=MAX_LENGTH,
    )

    model = BertOrdinalClareza(
        model_name=model_name,
        n_features=features_treino.shape[1],
        dropout=DROPOUT,
    )

    pos_weight = calcular_pos_weight(
        y_fold_treino
    )

    treinar_modelo(
        model=model,
        dataset=dataset_fold_treino,
        tokenizer=tokenizer,
        pos_weight=pos_weight,
        epochs=EPOCHS,
    )

    pred_treino = prever_indices(
        model,
        dataset_fold_treino,
        tokenizer,
        descricao=f"Fold {fold} - treino",
    )

    pred_validacao = prever_indices(
        model,
        dataset_fold_validacao,
        tokenizer,
        descricao=f"Fold {fold} - validação",
    )

    acc_treino = accuracy_score(
        y_fold_treino,
        pred_treino,
    )

    acc_validacao = accuracy_score(
        y_fold_validacao,
        pred_validacao,
    )

    tempo_fold = time.time() - inicio_fold

    resultados.append(
        {
            "fold": fold,
            "train_accuracy": acc_treino,
            "validation_accuracy": acc_validacao,
            "tempo_segundos": tempo_fold,
        }
    )

    log_resultado(
        f"Acurácia de treino: {acc_treino:.2%}"
    )

    log_resultado(
        f"Acurácia de validação: {acc_validacao:.2%}"
    )

    log_resultado(
        f"Tempo do fold: {tempo_fold / 60:.2f} min"
    )

    del model
    del tokenizer
    del dataset_fold_treino
    del dataset_fold_validacao

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

resultados_df = pd.DataFrame(resultados)

acuracia_validacao = (
    resultados_df["validation_accuracy"].mean()
)

acuracia_treino_cv = (
    resultados_df["train_accuracy"].mean()
)

gap = (
    acuracia_treino_cv
    - acuracia_validacao
)

tempo_cv = time.time() - inicio_cv

log_resultado()
log_resultado("=" * 60)
log_resultado("RESUMO DA VALIDAÇÃO CRUZADA")
log_resultado("=" * 60)
log_resultado(resultados_df.to_string(index=False))
log_resultado()
log_resultado(
    f"Acurácia Média da Validação Cruzada: "
    f"{acuracia_validacao:.2%}"
)
log_resultado(
    f"Acurácia Média de Treino nos Folds: "
    f"{acuracia_treino_cv:.2%}"
)
log_resultado(
    f"Gap Treino-Validação: "
    f"{gap:.2%}"
)
log_resultado(
    f"Tempo total da validação cruzada: "
    f"{tempo_cv / 60:.2f} min"
)

## Treino do modelo final

Depois da validação, o modelo é treinado com todo o conjunto de treino.

Se `USAR_TAPT=True`, a adaptação final usa `train.xlsx + test1.xlsx` sem rótulos antes do treinamento supervisionado.

In [ ]:
inicio_treino_final = time.time()

scaler_final = StandardScaler()

features_treino_final = scaler_final.fit_transform(
    features_treino
)

features_teste_final = scaler_final.transform(
    features_teste
)

model_name_final = MODELO_BASE

if USAR_TAPT:
    model_name_final = executar_tapt(
        textos=X_treino + X_teste,
        output_dir="modelos/tapt_final",
    )

tokenizer_final = AutoTokenizer.from_pretrained(
    model_name_final
)

dataset_treino_final = ClarezaDataset(
    textos=X_treino,
    features=features_treino_final,
    labels=y_indices,
    tokenizer=tokenizer_final,
    max_length=MAX_LENGTH,
)

model_final = BertOrdinalClareza(
    model_name=model_name_final,
    n_features=features_treino.shape[1],
    dropout=DROPOUT,
)

pos_weight_final = calcular_pos_weight(
    y_indices
)

treinar_modelo(
    model=model_final,
    dataset=dataset_treino_final,
    tokenizer=tokenizer_final,
    pos_weight=pos_weight_final,
    epochs=EPOCHS,
)

pred_treino_final = prever_indices(
    model_final,
    dataset_treino_final,
    tokenizer_final,
    descricao="Treino final - avaliação",
)

acc_treino_final = accuracy_score(
    y_indices,
    pred_treino_final,
)

tempo_treino_final = time.time() - inicio_treino_final

log_resultado()
log_resultado("=" * 60)
log_resultado("TREINO FINAL")
log_resultado("=" * 60)
log_resultado(
    f"Acurácia do Treino (modelo final): "
    f"{acc_treino_final:.2%}"
)
log_resultado(
    f"Tempo do treino final: "
    f"{tempo_treino_final / 60:.2f} min"
)

## Execução do teste e salvamento do resultado

A ordem das linhas de `test1.xlsx` é preservada. Antes de salvar, fazemos duas verificações simples para evitar erro de formato.

In [ ]:
dataset_teste = ClarezaDataset(
    textos=X_teste,
    features=features_teste_final,
    labels=None,
    tokenizer=tokenizer_final,
    max_length=MAX_LENGTH,
)

pred_teste_indices = prever_indices(
    model_final,
    dataset_teste,
    tokenizer_final,
    descricao="Teste",
)

Y_previsto = [
    INDEX_TO_LABEL[int(indice)]
    for indice in pred_teste_indices
]

assert len(Y_previsto) == len(X_teste)
assert set(Y_previsto).issubset(
    {"c1", "c234", "c5"}
)

distribuicao_teste = pd.Series(Y_previsto).value_counts()

log_resultado()
log_resultado("=" * 60)
log_resultado("DISTRIBUIÇÃO DAS PREVISÕES DO TESTE")
log_resultado("=" * 60)
log_resultado(distribuicao_teste.to_string())

distribuicao_teste

In [ ]:
teste_rotulado = pd.DataFrame(
    {
        "resp_text": X_teste,
        "clarity": Y_previsto,
    }
)

teste_rotulado.to_excel(
    "teste_rotulado.xlsx",
    index=False,
)

log_resultado()
log_resultado(f"Arquivo de teste salvo: teste_rotulado.xlsx")
log_resultado(f"Arquivo de log salvo: {LOG_PATH}")
log_resultado(
    f"Fim: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"
)

teste_rotulado.head()

## Observações para os próximos testes

Este experimento testa a hipótese de underfitting observada na execução anterior:

- `N_SPLITS = 3`;
- `EPOCHS = 2`;
- `MAX_LENGTH = 128`;
- TAPT desligado;
- padding dinâmico por batch;
- barras de progresso no treino e na predição;
- métricas salvas em `resultados_ep.txt`.

O objetivo imediato é comparar a acurácia e o gap com a execução de 1 época antes de alterar outros hiperparâmetros.

## Log do experimento

A célula abaixo mostra o mesmo conteúdo salvo em `resultados_ep.txt`.

In [ ]:
with open(LOG_PATH, "r", encoding="utf-8") as arquivo:
    print(arquivo.read())